In [ ]:
# --- utils.py embarque (base64) ------------------------------------------

import base64, os

_b64 = r"""..."""

_utils_bytes = base64.b64decode(_b64)

os.makedirs('/kaggle/working', exist_ok=True)

_utils_path = '/kaggle/working/utils.py'

with open(_utils_path, 'wb') as _f:

    _f.write(_utils_bytes)

# --- nib.load flexible : accepte .nii.gz ou .nii -------------------------

import nibabel as _nib

_orig_nib_load = _nib.load

def _nib_load_flexible(path, *a, **k):

    p = str(path)

    if p.endswith('.nii.gz'):

        alt = p[:-3]

        from pathlib import Path as _P

        if not _P(p).is_file() and _P(alt).is_file():

            path = alt

    return _orig_nib_load(path, *a, **k)

_nib.load = _nib_load_flexible

print(f"utils.py ecrit dans {_utils_path} OK")



# 02 Pretraitement



In [ ]:
# --- Charger utils comme module (robuste sous papermill) -----------------

import sys, types

_utils_mod = types.ModuleType('utils')

_utils_mod.__file__ = '/kaggle/working/utils.py'

exec(open('/kaggle/working/utils.py').read(), _utils_mod.__dict__)

sys.modules['utils'] = _utils_mod

utils = _utils_mod

# --- Configuration -------------------------------------------------------
import os, json
from pathlib import Path
import numpy as np
import nibabel as nib
import pandas as pd
from tqdm.auto import tqdm

if Path('/kaggle/working').is_dir():
    BASE = Path('/kaggle/working')
else:
    BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ROOT = utils.find_bhsd_root()
SPLITS = utils.load_splits(root=ROOT)
OUT = BASE / 'preprocessed'
OUT.mkdir(parents=True, exist_ok=True)
print('BHSD   :', ROOT)
print('Sorties :', OUT)


In [ ]:
# --- Charger utils comme module (robuste sous papermill) -----------------

import sys, types

_utils_mod = types.ModuleType('utils')

_utils_mod.__file__ = '/kaggle/working/utils.py'

exec(open('/kaggle/working/utils.py').read(), _utils_mod.__dict__)

sys.modules['utils'] = _utils_mod

utils = _utils_mod

# --- Listes train/val/test ------------------------------
for nom in ['train', 'val', 'test']:
    with open(OUT / f'{nom}_list.json', 'w', encoding='utf-8') as f:
        json.dump(SPLITS[nom], f, ensure_ascii=False, indent=2)
    print(f'{nom:5s} : {len(SPLITS[nom]):3d} scanners')
assert len(SPLITS['train']) == 80
assert len(SPLITS['val']) == 16
assert len(SPLITS['test']) == 96


In [ ]:
# --- Charger utils comme module (robuste sous papermill) -----------------

import sys, types

_utils_mod = types.ModuleType('utils')

_utils_mod.__file__ = '/kaggle/working/utils.py'

exec(open('/kaggle/working/utils.py').read(), _utils_mod.__dict__)

sys.modules['utils'] = _utils_mod

utils = _utils_mod

# --- Index des coupes (socle) -------------------------------------------
index_coupes_socle = {}
stats_socle = []
for cid in tqdm(SPLITS['train'], desc='Index socle'):
    gt = nib.load(ROOT / 'ground truths' / f'{cid}.nii.gz')
    mask = np.asarray(gt.dataobj).astype(np.int16)
    has_mask = np.any(mask != 0, axis=(0, 1))
    indices = np.where(has_mask)[0].tolist()
    index_coupes_socle[cid] = indices
    stats_socle.append({'scanner': cid, 'n_coupes_total': mask.shape[2], 'n_coupes_avec_mask': len(indices)})

with open(OUT / 'index_coupes_socle.json', 'w', encoding='utf-8') as f:
    json.dump(index_coupes_socle, f, ensure_ascii=False, indent=2)
print(f'index_coupes_socle.json : {len(index_coupes_socle)} scanners')


In [ ]:
# --- Charger utils comme module (robuste sous papermill) -----------------

import sys, types

_utils_mod = types.ModuleType('utils')

_utils_mod.__file__ = '/kaggle/working/utils.py'

exec(open('/kaggle/working/utils.py').read(), _utils_mod.__dict__)

sys.modules['utils'] = _utils_mod

utils = _utils_mod

# --- Index Lot G (coupes avec hemorragie < 5 mL) -------------------------
def volume_par_coupe(mask_3d, voxel_volume_ml):
    volumes = np.zeros(mask_3d.shape[2], dtype=np.float32)
    for z in range(mask_3d.shape[2]):
        n_vox = int(np.count_nonzero(mask_3d[:, :, z]))
        if n_vox > 0:
            volumes[z] = n_vox * voxel_volume_ml
    return volumes

index_coupes_lotg = {}
stats_lotg = []
for cid in tqdm(SPLITS['train'], desc='Index Lot G'):
    gt = nib.load(ROOT / 'ground truths' / f'{cid}.nii.gz')
    mask = np.asarray(gt.dataobj).astype(np.int16)
    voxel_vol_ml = float(np.prod(gt.header.get_zooms()[:3])) / 1000.0
    vol_par_z = volume_par_coupe(mask, voxel_vol_ml)
    indices_petites = np.where((vol_par_z > 0) & (vol_par_z < 5.0))[0].tolist()
    indices_toutes = np.where(vol_par_z > 0)[0].tolist()
    index_coupes_lotg[cid] = {'toutes_avec_mask': indices_toutes, 'petites_hemo_lt_5ml': indices_petites}
    stats_lotg.append({'scanner': cid, 'n_coupes_total': mask.shape[2], 'n_coupes_avec_mask': len(indices_toutes), 'n_coupes_petites_lt_5ml': len(indices_petites)})

with open(OUT / 'index_coupes_lotg.json', 'w', encoding='utf-8') as f:
    json.dump(index_coupes_lotg, f, ensure_ascii=False, indent=2)
df_lotg = pd.DataFrame(stats_lotg)
print(df_lotg.describe())
print(f'Total coupes petites (<5 mL) : {df_lotg["n_coupes_petites_lt_5ml"].sum()}')


In [ ]:
# --- Charger utils comme module (robuste sous papermill) -----------------

import sys, types

_utils_mod = types.ModuleType('utils')

_utils_mod.__file__ = '/kaggle/working/utils.py'

exec(open('/kaggle/working/utils.py').read(), _utils_mod.__dict__)

sys.modules['utils'] = _utils_mod

utils = _utils_mod

# --- Verification Lot G --------------------------------------------------
total_petites = df_lotg['n_coupes_petites_lt_5ml'].sum()
total_coupes_mask = df_lotg['n_coupes_avec_mask'].sum()
print(f'Coupes petites < 5 mL : {total_petites} / {total_coupes_mask} ({100*total_petites/total_coupes_mask:.1f}%)')
scanners_peu = df_lotg[df_lotg['n_coupes_petites_lt_5ml'] < 4]
print(f'Scanners avec < 4 coupes petites : {len(scanners_peu)} / {len(df_lotg)}')
